# Execution Backends

> nbdev source for local execution adapter and Slurm/Kubernetes mapping helpers.


In [ ]:
#| default_exp execution_backends


In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable
from uuid import uuid4

from ml_deploy.vertical_slice import execute_first_vertical_slice
from ml_deploy.webui_contracts import NotebookExecutionRequest, RunVisibility, summarize_run_for_webui


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


@dataclass(frozen=True)
class LocalExecutionResult:
    request_id: str
    run_visibility: RunVisibility
    output_dir: Path
    prediction_log_path: Path


class LocalExecutionAdapter:
    """Adapter that maps normalized web UI requests to local execution."""

    def __init__(self, mlflow_base_url: str, runner: Callable[..., dict[str, Any]] | None = None):
        self.mlflow_base_url = mlflow_base_url
        self._runner = runner or execute_first_vertical_slice

    def submit(
        self,
        request: NotebookExecutionRequest,
        *,
        work_dir: Path,
        tracking_uri: str,
        request_id: str | None = None,
    ) -> LocalExecutionResult:
        resolved_request = request.with_defaults(request_id=request_id or f"req-{uuid4().hex[:12]}")
        job_spec = resolved_request.as_job_spec()

        run_payload = self._runner(work_dir, tracking_uri=tracking_uri)
        run_id = run_payload["training_outcome"].mlflow_run_id
        run_summary = summarize_run_for_webui(
            {
                "run_id": run_id,
                "status": "finished",
                "started_at": _utc_now(),
                "finished_at": _utc_now(),
                "model_version": run_payload["packaged"]["model_version_record"][
                    "model_version_identifier"
                ],
                "request_id": job_spec["request_id"],
            },
            mlflow_base_url=self.mlflow_base_url,
        )

        return LocalExecutionResult(
            request_id=job_spec["request_id"],
            run_visibility=run_summary,
            output_dir=run_payload["output_dir"],
            prediction_log_path=run_payload["prediction_log_path"],
        )


def map_to_slurm_job_spec(job_spec: dict[str, Any]) -> dict[str, Any]:
    """Map normalized job spec to a Slurm submission payload."""
    notebook = job_spec["notebook"]
    return {
        "scheduler": "slurm",
        "job_name": f"nb-{job_spec['request_id']}",
        "command": "python -m ml_deploy.jobs.run_notebook",
        "arguments": [
            f"--repository={notebook['repository']}",
            f"--notebook-path={notebook['notebook_path']}",
            f"--revision={notebook['revision']}",
            f"--target={job_spec['target']}",
            f"--config-profile={job_spec['config_profile']}",
        ],
        "environment": {
            "REQUEST_ID": job_spec["request_id"],
            "REQUESTED_BY": job_spec["requested_by"],
        },
    }


def map_to_kubernetes_job_spec(job_spec: dict[str, Any]) -> dict[str, Any]:
    """Map normalized job spec to a Kubernetes Job manifest."""
    notebook = job_spec["notebook"]
    return {
        "apiVersion": "batch/v1",
        "kind": "Job",
        "metadata": {"name": f"nb-{job_spec['request_id']}"},
        "spec": {
            "template": {
                "spec": {
                    "restartPolicy": "Never",
                    "containers": [
                        {
                            "name": "notebook-runner",
                            "image": "python:3.13-slim",
                            "args": [
                                "python",
                                "-m",
                                "ml_deploy.jobs.run_notebook",
                                f"--repository={notebook['repository']}",
                                f"--notebook-path={notebook['notebook_path']}",
                                f"--revision={notebook['revision']}",
                                f"--target={job_spec['target']}",
                                f"--config-profile={job_spec['config_profile']}",
                            ],
                            "env": [
                                {"name": "REQUEST_ID", "value": job_spec["request_id"]},
                                {"name": "REQUESTED_BY", "value": job_spec["requested_by"]},
                            ],
                        }
                    ],
                }
            }
        },
    }
